# 00_setup - Aprovisionamiento DEV FinPay Lakehouse

Este notebook prepara el ambiente **DEV**.

Criterios:
- Catálogo: `fintech_finpay_dev`.
- Bronze: columnas de negocio en `STRING`.
- Silver: columnas tipadas según semántica.
- `ingestion_archetypes.json` generado dinámicamente para DEV.
- `silver.users` protegido con column masking y row-level security.


In [ ]:

ENV = "dev"
CATALOG = "fintech_finpay_dev"

SCHEMA_DEFAULT = "default"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"
SCHEMA_OBSERVABILITY = "observability"

VOLUME_NAME = "vol_landing"
LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA_DEFAULT}/{VOLUME_NAME}"

GROUP_INGENIERIA = "finpay_ingenieria"
GROUP_RIESGO = "finpay_riesgo"
GROUP_AUDITORIA = "finpay_auditoria"

CURRENT_USER = spark.sql("SELECT current_user() AS user").first()["user"]

print(f"ENV          : {ENV}")
print(f"CATALOG      : {CATALOG}")
print(f"LANDING_PATH : {LANDING_PATH}")
print(f"CURRENT_USER : {CURRENT_USER}")


In [ ]:

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

for schema_name in [SCHEMA_DEFAULT, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_OBSERVABILITY]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}")

spark.sql(f'''
CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME}
COMMENT 'Landing zone DEV para archivos fuente del proyecto FinPay'
''')

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA_DEFAULT}"))


In [ ]:


directories = [
    f"{LANDING_PATH}/transactions",
    f"{LANDING_PATH}/merchants",
    f"{LANDING_PATH}/users",
    f"{LANDING_PATH}/metadata",
    f"{LANDING_PATH}/metadata/schema",
    f"{LANDING_PATH}/metadata/schema/transactions",
    f"{LANDING_PATH}/metadata/schema/merchants",
    f"{LANDING_PATH}/metadata/schema/users",
    f"{LANDING_PATH}/metadata/checkpoints",
    f"{LANDING_PATH}/metadata/checkpoints/transactions",
    f"{LANDING_PATH}/metadata/checkpoints/merchants",
    f"{LANDING_PATH}/metadata/checkpoints/users",
    f"{LANDING_PATH}/quarantine",
]

for path in directories:
    dbutils.fs.mkdirs(path)

display(dbutils.fs.ls(LANDING_PATH))


In [ ]:
# COMMAND ----------
# DBTITLE 1,Generar ingestion_archetypes.json DEV

import json

ingestion_archetypes = [
    {
        "source_name": "transactions",
        "source_path": f"{LANDING_PATH}/transactions/",
        "file_format": "csv",
        "delimiter": ",",
        "header": True,
        "multiline": False,
        "schema_location": f"{LANDING_PATH}/metadata/schema/transactions/",
        "checkpoint_path": f"{LANDING_PATH}/metadata/checkpoints/transactions/",
        "partition_by": "transaction_date",
        "target_table": f"{CATALOG}.{SCHEMA_BRONZE}.transactions",
        "active": True
    },
    {
        "source_name": "merchants",
        "source_path": f"{LANDING_PATH}/merchants/",
        "file_format": "json",
        "delimiter": None,
        "header": None,
        "multiline": True,
        "schema_location": f"{LANDING_PATH}/metadata/schema/merchants/",
        "checkpoint_path": f"{LANDING_PATH}/metadata/checkpoints/merchants/",
        "partition_by": "country",
        "target_table": f"{CATALOG}.{SCHEMA_BRONZE}.merchants",
        "active": True
    },
    {
        "source_name": "users",
        "source_path": f"{LANDING_PATH}/users/",
        "file_format": "csv",
        "delimiter": "|",
        "header": True,
        "multiline": False,
        "schema_location": f"{LANDING_PATH}/metadata/schema/users/",
        "checkpoint_path": f"{LANDING_PATH}/metadata/checkpoints/users/",
        "partition_by": "country",
        "target_table": f"{CATALOG}.{SCHEMA_BRONZE}.users",
        "active": True
    }
]

metadata_path = f"{LANDING_PATH}/metadata/ingestion_archetypes.json"

dbutils.fs.put(
    metadata_path,
    json.dumps(ingestion_archetypes, indent=2, ensure_ascii=False),
    overwrite=True
)

print(f"Archivo creado: {metadata_path}")
print(dbutils.fs.head(metadata_path, 5000))


In [ ]:
# COMMAND ----------
# DBTITLE 1,Asignar permisos por rol

def grant(sql_statement: str):
    print(sql_statement)
    spark.sql(sql_statement)

for principal in [GROUP_INGENIERIA, GROUP_RIESGO, GROUP_AUDITORIA]:
    grant(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{principal}`")

for schema_name in [SCHEMA_DEFAULT, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_OBSERVABILITY]:
    grant(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT CREATE TABLE ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT CREATE FUNCTION ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT MODIFY ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT SELECT ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")

for schema_name in [SCHEMA_SILVER, SCHEMA_GOLD]:
    grant(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_RIESGO}`")
    grant(f"GRANT SELECT ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_RIESGO}`")

for schema_name in [SCHEMA_GOLD, SCHEMA_OBSERVABILITY]:
    grant(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_AUDITORIA}`")
    grant(f"GRANT SELECT ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_AUDITORIA}`")

grant(f"GRANT READ VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_INGENIERIA}`")
grant(f"GRANT WRITE VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_INGENIERIA}`")

grant(f"GRANT READ VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_RIESGO}`")
grant(f"GRANT READ VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_AUDITORIA}`")


In [ ]:
# COMMAND ----------
# DBTITLE 1,Validaciones finales

print("=== Schemas ===")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

print("=== Landing folders ===")
display(dbutils.fs.ls(LANDING_PATH))

print("=== ingestion_archetypes.json ===")
print(dbutils.fs.head(f"{LANDING_PATH}/metadata/ingestion_archetypes.json", 5000))

print("=== Bronze tables ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_BRONZE}"))

print("=== Silver tables ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_SILVER}"))

print("=== Observability tables ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_OBSERVABILITY}"))

print("Setup DEV finalizado correctamente.")
